# Step 2: Characterization — Background & PSF Modeling

After ISR, the image contains sky + sources. Before measuring sources, we need to:
1. **Estimate and subtract the sky background**
2. **Model the PSF** — the single most critical step for weak lensing

**LSST task:** `lsst.pipe.tasks.characterizeImage.CharacterizeImageTask`  
**Input:** `postISRCCD`  
**Output:** `icExp` (characterized exposure with PSF model attached)

**Reference:** Bosch et al. (2018) §3.3–3.4

## 2.1 Background Estimation

### The problem
The sky is not dark — scattered moonlight, zodiacal light, atmospheric
emission lines (OH), and instrumental glow all contribute a smooth but
spatially-varying background. This must be subtracted so that source
flux measurements are unbiased.

### The algorithm
1. Divide the image into a grid of **superpixels** (default: 128×128 pixels)
2. In each superpixel, estimate the sky as the **clipped mean** or **median**
   (iteratively rejecting pixels with sources)
3. Fit a smooth 2D model (Chebyshev polynomial or spline) to the grid
4. Subtract the smooth model from the image

### Why it matters for shapes
Background over-subtraction near large galaxies clips their outer isophotes,
biasing shapes toward rounder. Under-subtraction adds flux that dilutes
the galaxy's ellipticity signal. The HSC pipeline uses a 6th-order
Chebyshev polynomial on 128-pixel bins (Bosch et al. 2018).

```python
# In the LSST pipeline:
config.background.binSize = 128         # superpixel size
config.background.algorithm = 'NATURAL_SPLINE'
config.background.statisticsProperty = 'MEANCLIP'
```

## 2.2 PSF Modeling

### Why the PSF is everything

The PSF is the image of a point source. Every galaxy image is the **true
galaxy light profile convolved with the PSF**:

$$I^{\text{obs}}(\vec{x}) = I^{\text{true}}(\vec{x}) * \text{PSF}(\vec{x}) + \text{noise}$$

The PSF has two devastating effects on shape measurement:

1. **Circularization (isotropic PSF):** A round PSF smears galaxies,
   making them rounder → dilutes the shear signal
2. **Spurious ellipticity (anisotropic PSF):** An elliptical PSF imprints
   its shape onto galaxies → creates a fake shear signal

Both effects must be corrected, which requires an accurate PSF model
at every position on the focal plane.

### What determines the PSF?
- **Atmosphere:** Turbulence creates a time-varying, seeing-limited PSF
  (~0.6–1.2" FWHM). Varies on seconds–minutes timescales.
- **Optics:** Aberrations (coma, astigmatism, defocus) vary across the
  field of view. Static or slowly-varying.
- **Detector:** Charge diffusion, brighter-fatter (corrected in ISR),
  pixel response non-uniformity.
- **Tracking:** Wind shake and tracking errors broaden the PSF.

The combined PSF varies smoothly across the focal plane within a single
exposure, but changes from exposure to exposure.

### PSF Modeling Algorithms

#### PSFEx (default in DP0.2)
`lsst.meas.extensions.psfex`

1. Select bright, unsaturated, isolated stars
2. Represent PSF as a sum of pixel-basis functions
3. Model spatial variation as polynomial in (x, y) focal-plane coordinates
4. Solve via least-squares

#### PIFF (preferred for Rubin)
`lsst.meas.extensions.piff`

Developed by DES. Supports multiple basis types (pixel, shapelet, Gaussian).
Better handling of CCD boundaries and discontinuities.

### Star Selection
Stars are selected by their location in the **size-magnitude diagram**:
stars form a tight locus at constant size (= PSF size), while galaxies
are resolved and larger.

The HSC pipeline uses k-means clustering to separate the stellar locus.
Typically ~72 stars per CCD are used for PSF fitting.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

rng = np.random.default_rng(42)

In [ ]:
# --- Simulate background estimation ---

ny, nx = 256, 256
yy, xx = np.mgrid[:ny, :nx]

# Smooth sky background (moonlight gradient + atmospheric emission)
sky_bg = 300 + 50 * (xx / nx) + 30 * np.sin(2 * np.pi * yy / ny * 0.5)

# Add a few sources
sources = np.zeros((ny, nx))
for _ in range(10):
    cx, cy = rng.integers(30, nx-30), rng.integers(30, ny-30)
    flux = 10**rng.uniform(3, 4.5)
    sigma = rng.uniform(2, 6)
    sources += flux * np.exp(-((xx-cx)**2 + (yy-cy)**2) / (2*sigma**2)) / (2*np.pi*sigma**2)

# Observed = sky + sources + noise
observed = sky_bg + sources + rng.normal(scale=15, size=(ny, nx))

# Estimate background: bin into 32x32 superpixels, take clipped median
bin_size = 32
bg_estimate = np.zeros_like(observed)
for iy in range(0, ny, bin_size):
    for ix in range(0, nx, bin_size):
        patch = observed[iy:iy+bin_size, ix:ix+bin_size]
        # Iterative sigma clipping (simple version)
        vals = patch.ravel()
        for _ in range(3):
            med = np.median(vals)
            std = np.std(vals)
            vals = vals[np.abs(vals - med) < 3 * std]
        bg_estimate[iy:iy+bin_size, ix:ix+bin_size] = np.median(vals)

# Smooth the estimate (in practice: spline or Chebyshev fit)
from scipy.ndimage import gaussian_filter
bg_smooth = gaussian_filter(bg_estimate, sigma=16)

bg_subtracted = observed - bg_smooth

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(observed, cmap='gray', origin='lower')
axes[0].set_title('Observed (sky + sources)')

axes[1].imshow(bg_smooth, cmap='viridis', origin='lower')
axes[1].set_title('Estimated background')

axes[2].imshow(sky_bg, cmap='viridis', origin='lower')
axes[2].set_title('True background')

vmin = np.percentile(sources, 1)
vmax = np.percentile(sources, 99.5)
axes[3].imshow(bg_subtracted, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[3].set_title('Background-subtracted')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Background Estimation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/background_estimation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Simulate PSF variation across the focal plane ---

def make_psf(fwhm, e1, e2, stamp_size=31):
    """Create an elliptical Gaussian PSF."""
    sigma = fwhm / 2.355
    y, x = np.mgrid[:stamp_size, :stamp_size] - stamp_size // 2
    # Convert ellipticity to covariance matrix
    # Q = sigma^2 * [[1+e1, e2], [e2, 1-e1]]
    Qxx = sigma**2 * (1 + e1)
    Qyy = sigma**2 * (1 - e1)
    Qxy = sigma**2 * e2
    det = Qxx * Qyy - Qxy**2
    psf = np.exp(-0.5 * (Qyy * x**2 - 2*Qxy*x*y + Qxx*y**2) / det)
    return psf / psf.sum()


# PSF varies across a 4x4 grid (simulating a focal plane)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))

for iy in range(4):
    for ix in range(4):
        # PSF properties vary with position
        # Seeing varies slightly
        fwhm = 3.5 + 0.3 * np.sqrt((ix-1.5)**2 + (iy-1.5)**2)
        # Ellipticity from optics (radial pattern = astigmatism)
        r = np.sqrt((ix-1.5)**2 + (iy-1.5)**2)
        theta = np.arctan2(iy-1.5, ix-1.5)
        e_mag = 0.03 * r  # ellipticity grows with distance from center
        e1 = e_mag * np.cos(2*theta)
        e2 = e_mag * np.sin(2*theta)

        psf = make_psf(fwhm, e1, e2)
        axes[iy, ix].imshow(psf, cmap='inferno', origin='lower')
        axes[iy, ix].set_xticks([]); axes[iy, ix].set_yticks([])
        axes[iy, ix].set_title(f'e={e_mag:.2f}', fontsize=9)

plt.suptitle('PSF Variation Across Focal Plane\n'
             '(size increases toward edges, ellipticity shows astigmatic pattern)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/psf_variation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Star-galaxy separation: the size-magnitude diagram ---

n_stars = 200
n_gals = 500

# Stars: constant size (= PSF FWHM), vary in brightness
star_mag = rng.uniform(18, 24, n_stars)
star_size = 3.5 + 0.15 * rng.normal(size=n_stars)  # tight locus at PSF size

# Galaxies: resolved, so larger than PSF; size increases for brighter ones
gal_mag = rng.uniform(20, 26, n_gals)
gal_size = 3.5 + rng.exponential(1.5, n_gals)  # always >= PSF size

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(star_mag, star_size, s=10, alpha=0.6, label='Stars', color='gold')
ax.scatter(gal_mag, gal_size, s=10, alpha=0.4, label='Galaxies', color='steelblue')
ax.axhline(3.5, color='red', ls='--', lw=1.5, label='PSF FWHM')
ax.set_xlabel('Magnitude', fontsize=12)
ax.set_ylabel('Measured FWHM (pixels)', fontsize=12)
ax.set_title('Size–Magnitude Diagram (Star-Galaxy Separation)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(17, 27)
ax.set_ylim(2, 12)
ax.invert_xaxis()
plt.tight_layout()
plt.savefig('../../figures/size_magnitude.png', dpi=150, bbox_inches='tight')
plt.show()

print("Stars form a tight horizontal locus at the PSF size.")
print("These are selected and used to build the PSF model.")
print("Galaxies scatter above — they are resolved (larger than the PSF).")

## PSF Quality Metrics

After building the PSF model, we validate it by checking:

1. **Residual size:** `(T_star - T_model) / T_star` where T = trace of moments
   - Requirement: < 0.001 (fractional size residual)

2. **Residual ellipticity:** `e_star - e_model`
   - Requirement: < 0.0002 per component

3. **ρ-statistics (Rowe statistics):** Correlation functions of PSF model
   residuals that directly propagate into shear two-point functions

```python
# In LSST pipeline outputs:
# Compare PSF model moments to reserved star moments
T_star = sources['base_SdssShape_xx'] + sources['base_SdssShape_yy']
T_psf = sources['base_SdssShape_psf_xx'] + sources['base_SdssShape_psf_yy']
delta_T_frac = (T_star - T_psf) / T_star  # should be ~0

e1_star = sources['base_SdssShape_xx'] - sources['base_SdssShape_yy']
e1_psf = sources['base_SdssShape_psf_xx'] - sources['base_SdssShape_psf_yy']
delta_e1 = (e1_star - e1_psf) / T_star  # should be ~0
```

## Summary

| Component | Algorithm | Key parameter | Output |
|-----------|-----------|---------------|--------|
| Background | Binned clipped-mean + spline | `binSize=128` | Smooth sky model |
| Star selection | Size-magnitude clustering | ~72 stars/CCD | Star catalog |
| PSF model | PSFEx or PIFF | Polynomial order for spatial variation | PSF at any (x,y) |

The PSF model is attached to the Exposure object and travels with it
through all subsequent processing. Every measurement plugin can evaluate
`exposure.getPsf().computeImage(position)` to get the local PSF.

**Next:** [04_calibration.ipynb](04_calibration.ipynb) — Astrometric and photometric calibration